# Experimento D1 · Escalabilidad en volumen
**Responsable:** Andrade

## 1. Objetivo
Medir cómo cambian la latencia y el throughput al calcular los indicadores I1, I2 e I3 con 1, 5 y 10 millones de filas.

**Hipótesis:** Al multiplicar el volumen por 10, la latencia aumenta menos de 10 veces, porque Spark reparte el trabajo en paralelo.

## 2. Configuración y condiciones de prueba
- Cómputo: el de tu workspace de Databricks (anótalo en el informe).
- Tablas Delta: `silver_nacimientos_1x` (1,048,575), `_5x` (5,048,575), `_10x` (10,048,575).
- Consulta: KPIs agrupados por año, materializados con `collect()`.

## 3. Procedimiento
1. Ejecuta `%run ./00_utilidades`.
2. Ejecuta las celdas de medición en orden. Cada medida hace **1 ejecución de calentamiento (no se cuenta) y 3 repeticiones**.
3. Los resultados se guardan solos en la tabla `resultados_experimentos`.
4. Completa el análisis (sección 5) con tus números.

**Evidencias a capturar:** la celda de medición con su salida, y la tabla de resultados.

In [ ]:
%run ./00_utilidades

In [ ]:
EXP, RESP = "D1_escalabilidad", "Andrade"
for esc, tabla in [("1x", T_SILVER_1X), ("5x", T_SILVER_5X), ("10x", T_SILVER_10X)]:
    df = leer_tabla(tabla)
    filas = df.count()
    tiempos = medir(lambda d=df: calcular_kpis(d, "anio").collect())
    f = registrar_resultado(EXP, f"volumen_{esc}", esc, filas, tiempos, "Spark", RESP,
                            "KPIs por año; 1 calentamiento + 3 repeticiones")
    print(f"{esc}: {filas:,} filas | {f.media_s} s (± {f.desv_s}) | {f.throughput_filas_s:,} filas/s")

## 4. Resultados

In [ ]:
res = leer_tabla(T_RESULTADOS).where("experimento = 'D1_escalabilidad'").orderBy("filas")
display(res)

In [ ]:
# Comparación contra la escala 1x
filas_res = res.collect()
base = filas_res[0]
for r in filas_res:
    print(f"{r.escala}: datos x{r.filas / base.filas:.1f} | tiempo x{r.media_s / base.media_s:.2f} | throughput x{r.throughput_filas_s / base.throughput_filas_s:.2f}")

## 5. Comparación y análisis (completa con tus números)
- ¿Cuántas veces creció la latencia cuando los datos crecieron 5 y 10 veces?
- ¿El throughput (filas/s) aumentó? ¿Por qué? (pista: parte del tiempo es fijo, el arranque del trabajo en Spark.)
- ¿Se cumplió la hipótesis? Escribe una conclusión de 3 líneas.

**Límite:** la evaluación cambia el volumen en un mismo cómputo; no mide el efecto de agregar nodos.